## 1. Setup e importaciones

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Agregar la carpeta scripts al path para importar resilience_lib
BASE_DIR = "/home/aninotna/magister/tesis/justh2_pipeline"
SCRIPTS_DIR = os.path.join(BASE_DIR, "scripts")
if SCRIPTS_DIR not in sys.path:
    sys.path.append(SCRIPTS_DIR)

# Importar funciones desde resilience_lib
from scripts.resilience_lib_2 import (
    # Componentes IRCT
    compute_IRCT_pixel_wise,
    compute_reconstruction_anomaly,
    compute_latent_displacement,
    compute_cluster_stability_softmax,
    compute_cluster_expansion,
    compute_h2_stability,
    percentile_rank,
    # Validaciones
    validate_scenario_coherence,
    validate_spatial_stability,
    validate_structural_stability,
    validate_robustness,
    compute_morans_i,
    # Utilidades
    component_alignment,
    precompute_irct_components,
    sample_dirichlet_around,
    topk_jaccard
)

print("✓ resilience_lib importada correctamente")
print(f"  Ubicación: {os.path.join(SCRIPTS_DIR, 'resilience_lib.py')}")

## 2. Carga de datos (simulado)

En un caso real, cargarías:
- `irct_results_dict`: dict con IRCT por modelo/escenario
- `clustering_results_dict`: dict con resultados clustering
- `coords_df`: DataFrame con coordenadas (lat, lon)
- `models`: dict con modelos PyTorch entrenados

Para este ejemplo, mostraremos cómo estructurar los datos.

In [1]:
# Simulación de datos para demostración
# En tu caso real, estos vendrían de:
# - Experimento 1: cálculo del IRCT
# - Clustering: resultados de KMeans/SOM

# Crear datos de ejemplo
n_samples = 661  # píxeles
k_clusters = 10  # número de clusters
model_names = ['AE', 'VAE']
scenarios = ['SSP245', 'SSP370', 'SSP585']

# Crear estructura de datos tipo IRCT_RESULTS
irct_results_dict = {}
for model in model_names:
    irct_results_dict[model] = {}
    for scenario in scenarios:
        # Simular componentes y IRCT
        np.random.seed(42)
        A = np.random.uniform(0.3, 0.95, n_samples)
        S_D = np.random.uniform(0.2, 0.9, n_samples)
        S_C = np.random.uniform(0.4, 1.0, n_samples)
        S_E = np.random.uniform(0.3, 0.95, n_samples)
        S_H2 = np.random.uniform(0.2, 0.85, n_samples)
        
        # Media geométrica ponderada
        w = {'w_a': 0.15, 'w_d': 0.30, 'w_c': 0.25, 'w_e': 0.20, 'w_h': 0.10}
        IRCT = (A**w['w_a']) * (S_D**w['w_d']) * (S_C**w['w_c']) * (S_E**w['w_e']) * (S_H2**w['w_h'])
        
        # Degradar IRCT según escenario (SSP245 > SSP370 > SSP585)
        if scenario == 'SSP370':
            IRCT = IRCT * 0.9  # 10% degradación
        elif scenario == 'SSP585':
            IRCT = IRCT * 0.75  # 25% degradación
        
        irct_results_dict[model][scenario] = {
            'IRCT': IRCT,
            'A': A,
            'S_D': S_D,
            'S_C': S_C,
            'S_E': S_E,
            'S_H2': S_H2,
            'weights': w
        }

print(f"✓ Datos IRCT simulados: {len(irct_results_dict)} modelos × {len(scenarios)} escenarios")
print(f"  Dimensiones: {n_samples} píxeles × {len(irct_results_dict[model_names[0]]['SSP245']['IRCT'])} muestras")

NameError: name 'np' is not defined

In [ ]:
# Crear datos de coordenadas
coords_df = pd.DataFrame({
    'lat': np.random.uniform(-33.27, -32.26, n_samples),
    'lon': np.random.uniform(-71.89, -70.00, n_samples)
})

# Crear datos de clustering (simulado)
clustering_results_dict = {}
for model in model_names:
    clustering_results_dict[model] = {
        'labels_B245': np.random.randint(0, k_clusters, n_samples),
        'labels_T245': np.random.randint(0, k_clusters, n_samples),
        'labels_B370': np.random.randint(0, k_clusters, n_samples),
        'labels_T370': np.random.randint(0, k_clusters, n_samples),
        'labels_B585': np.random.randint(0, k_clusters, n_samples),
        'labels_T585': np.random.randint(0, k_clusters, n_samples),
    }

print(f"✓ Datos de clustering simulados: {len(clustering_results_dict)} modelos × {k_clusters} clusters")
print(f"✓ Coordenadas: {len(coords_df)} píxeles")

## 3. Validación 1: Coherencia por escenario

In [ ]:
# Usar la función de librería
print("VALIDACIÓN 1: COHERENCIA POR ESCENARIO")
print("="*80)
print()

validation_scenario = validate_scenario_coherence(irct_results_dict, model_names)

for model_key, results in validation_scenario.items():
    print(f"{model_key}:")
    print(f"  Medianas: 245={results['median_245']:.4f}, 370={results['median_370']:.4f}, 585={results['median_585']:.4f}")
    print(f"  Orden correcto (245 > 370 > 585): {'✓' if results['order_correct'] else '✗'}")
    print(f"  Wilcoxon p (245>370): {results['wilcoxon_245_370_p']:.4e}")
    print(f"  KS p (245>585): {results['ks_245_585_p']:.4e}")
    print()

## 4. Validación 2: Estabilidad espacial (Moran's I)

In [ ]:
print("VALIDACIÓN 2: ESTABILIDAD ESPACIAL (MORAN'S I)")
print("="*80)
print()

# Usar la función de librería con parámetros personalizables
validation_spatial = validate_spatial_stability(irct_results_dict, coords_df, model_names)

for model_key, scenarios_dict in validation_spatial.items():
    print(f"{model_key}:")
    for scenario_name, spatial_data in scenarios_dict.items():
        print(f"  {scenario_name}:")
        print(f"    Moran's I: {spatial_data['morans_I']:.4f}")
        print(f"    p-value (permutaciones): {spatial_data['p_value_perm_one_sided_pos']:.4e}")
        print(f"    Significativo y positivo: {'✓' if spatial_data['is_significant_perm'] and spatial_data['is_positive'] else '✗'}")
    print()

## 5. Validación 3: Estabilidad estructural

In [ ]:
print("VALIDACIÓN 3: ESTABILIDAD ESTRUCTURAL")
print("="*80)
print()

validation_structural = validate_structural_stability(irct_results_dict, clustering_results_dict, model_names)

for model_key, scenarios_dict in validation_structural.items():
    print(f"{model_key}:")
    for scenario_name, structural_data in scenarios_dict.items():
        print(f"  {scenario_name}:")
        print(f"    Spearman ρ: {structural_data.get('ari', np.nan):.4f} (ARI)")
        print(f"    NMI: {structural_data.get('nmi', np.nan):.4f}")
        print(f"    Retención: {structural_data.get('retention_rate', np.nan)*100:.1f}%")
    print()

## 6. Validación 4: Robustez del índice

In [ ]:
print("VALIDACIÓN 4: ROBUSTEZ DEL ÍNDICE (perturbaciones de pesos)")
print("="*80)
print()

# Usar función de robustez con parámetros personalizables
validation_robustness = validate_robustness(
    irct_results_dict,
    comps_dict=None,  # Si no tienes componentes precomputados
    clustering_results_dict=clustering_results_dict,
    model_order=model_names,
    n_perturbations=30,  # Menos perturbaciones para demo (normalmente 50)
    perturbation_magnitude=0.20,
    seed=42
)

for model_key, scenarios_dict in validation_robustness.items():
    print(f"{model_key}:")
    for scenario_name, robustness_data in scenarios_dict.items():
        print(f"  {scenario_name}:")
        print(f"    Spearman ρ (perturbaciones): μ={robustness_data['spearman_mean']:.4f} ± {robustness_data['spearman_std']:.4f}")
        print(f"    Robust share (ρ>0.90, τ>0.80, J>0.70): {robustness_data['robust_share_all_thresholds']*100:.1f}%")
        print(f"    Índice robusto: {'✓ ROBUSTO' if robustness_data['is_robust'] else '✗ NO ROBUSTO'}")
    print()

## 7. Resumen consolidado de validaciones

In [ ]:
print("RESUMEN CONSOLIDADO DE VALIDACIONES")
print("="*80)
print()

# Crear tabla resumen
summary_rows = []
for model_key in model_names:
    for scenario in scenarios:
        scenario_val = validation_scenario[model_key]
        spatial_val = validation_spatial[model_key].get(scenario, {})
        structural_val = validation_structural[model_key].get(scenario, {})
        robustness_val = validation_robustness[model_key].get(scenario, {})
        
        row = {
            'Modelo': model_key,
            'Escenario': scenario,
            'IRCT_median': np.median(irct_results_dict[model_key][scenario]['IRCT']),
            'Orden_correcto': '✓' if scenario_val.get('order_correct', False) else '✗',
            'Morans_I': spatial_val.get('morans_I', np.nan),
            'Espacial_OK': '✓' if spatial_val.get('is_significant_perm', False) and spatial_val.get('is_positive', False) else '✗',
            'NMI': structural_val.get('nmi', np.nan),
            'Retención_%': structural_val.get('retention_rate', np.nan) * 100,
            'Robustez_ρ': robustness_val.get('spearman_mean', np.nan),
            'Robustez_OK': '✓' if robustness_val.get('is_robust', False) else '✗',
        }
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
print()

# Tasas de éxito
print("TASAS DE ÉXITO:")
n_total = len(df_summary)
print(f"  Orden correcto: {(df_summary['Orden_correcto'] == '✓').sum()}/{n_total} ({100*(df_summary['Orden_correcto'] == '✓').sum()/n_total:.1f}%)")
print(f"  Estabilidad espacial: {(df_summary['Espacial_OK'] == '✓').sum()}/{n_total} ({100*(df_summary['Espacial_OK'] == '✓').sum()/n_total:.1f}%)")
print(f"  Robustez: {(df_summary['Robustez_OK'] == '✓').sum()}/{n_total} ({100*(df_summary['Robustez_OK'] == '✓').sum()/n_total:.1f}%)")

## 8. Uso avanzado: Personalización de parámetros

In [ ]:
print("EJEMPLO: PERSONALIZACIÓN DE PARÁMETROS")
print("="*80)
print()

# Opción 1: Mayor precisión en Moran's I (más permutaciones)
print("1. Moran's I con mayor precisión (n_perm=1999):")
coords_array = coords_df[['lat', 'lon']].values
I, expected_I, z_norm, p_perm, p_norm = compute_morans_i(
    irct_results_dict['AE']['SSP245']['IRCT'],
    coords_array,
    k_neighbors=8,
    n_perm=1999,  # Mayor precisión
    use_normal_approx=True
)
print(f"  Moran's I: {I:.4f}")
print(f"  p-value (permutaciones, n=1999): {p_perm:.4e}")
print()

# Opción 2: Robustez más estricta (perturbación mayor)
print("2. Robustez con perturbación más estricta (±40%):")
from scripts.resilience_lib_2 import topk_jaccard

# Simular perturbación con mayor magnitud
rng = np.random.default_rng(42)
weights_original = irct_results_dict['AE']['SSP245']['weights']
weights_perturbed = sample_dirichlet_around(weights_original, perturbation=0.40, rng=rng)
print(f"  Pesos originales: {weights_original}")
print(f"  Pesos perturbados: {weights_perturbed}")
print()

# Opción 3: Análisis de alineación de componentes
print("3. Alineación de componentes con IRCT:")
comps = {
    'A': irct_results_dict['AE']['SSP245']['A'],
    'S_D': irct_results_dict['AE']['SSP245']['S_D'],
    'S_C': irct_results_dict['AE']['SSP245']['S_C'],
    'S_E': irct_results_dict['AE']['SSP245']['S_E'],
    'S_H2': irct_results_dict['AE']['SSP245']['S_H2']
}

align_stats = component_alignment(irct_results_dict['AE']['SSP245']['IRCT'], comps, k_ratio=0.10)
for comp_name, stats in align_stats.items():
    print(f"  {comp_name}: Spearman ρ={stats['spearman']:.4f}, Jaccard Top-10%={stats['topk_jaccard']:.4f}")

## 9. Exportación de resultados

In [ ]:
# Guardar tabla resumen en CSV
output_dir = os.path.join(BASE_DIR, "data", "autoencoder_results")
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "validation_summary_example.csv")
df_summary.to_csv(output_file, index=False)
print(f"✓ Resultados guardados en: {output_file}")
print()
print("Puedes ahora:")
print("  1. Compartir los resultados con otros notebooks")
print("  2. Comparar validaciones entre diferentes versiones del índice")
print("  3. Usar estas funciones para validar nuevos índices sin duplicar código")

## Conclusiones

### Ventajas de usar `resilience_lib`:

✓ **Modularidad**: Cada validación es una función independiente

✓ **Reutilización**: Importa en cualquier notebook sin duplicar código

✓ **Flexibilidad**: Personaliza parámetros según tu caso

✓ **Documentación**: Docstrings completos con explicaciones matemáticas

✓ **Mantenibilidad**: Cambios centralizados benefician a todos los usos

### Próximos pasos:

1. Usar `resilience_lib.compute_IRCT_pixel_wise()` para calcular nuevos índices
2. Validar con las 4 funciones de validación
3. Generar reportes consolidados automáticamente
4. Integrar en pipelines de análisis complejos